# Notebook 22 — بنچمارک روی Matbench (matbench_phonons)

## هدف
تا اینجا معماری Dual Graph GNN فقط روی دیتاست اختصاصی این پروژه (۳۵۸ ماده MAX Phase،
پیش‌بینی IFC کامل + فرکانس فونون از طریق Phonopy) تست شده. این notebook یک مسیر **موازی
و مستقل** است: تست همون معماری پایه (بدون بخش فیزیکی/Phonopy، چون لازم نیست) روی
تسک رسمی **`matbench_phonons`** از لیدربورد Matbench، تا معلوم بشه در مقایسه با
مدل‌های منتشرشده کجای لیدربورد قرار می‌گیریم.

## تفاوت مهم این تسک با پروژه‌ی اصلی
| | پروژه‌ی اصلی (MAX Phase) | matbench_phonons |
|---|---|---|
| خروجی | کل ماتریس IFC → منحنی dispersion کامل (از طریق Phonopy) | **یک عدد اسکالر**: فرکانس پیک آخر phonon DOS |
| واحد | THz | **cm⁻¹** |
| تعداد نمونه | 358 | **1265** |
| نیاز به Phonopy در inference | بله | **خیر** (چون خروجی مستقیماً یک عدد رگرسیونی است) |
| پروتکل ارزیابی | Train/Val/Test ساده (seed=42) | **nested 5-fold CV رسمی Matbench** |

چون خروجی اینجا فقط یک عدد است (نه ماتریس IFC)، نیازی به قیدهای فیزیکی (symmetry، ASR)
یا موتور Phonopy نیست — فقط از انکودر دوگانه‌ی گراف (bond graph + atom graph) پروژه به‌عنوان
یک feature extractor استفاده می‌شود و یک سر رگرسیون ساده روی آن می‌نشیند.

## معیار مقایسه (جمع‌آوری‌شده از لیدربورد و مقالات مرتبط)

| مدل | MAE (cm⁻¹) | یادداشت |
|---|---|---|
| CGCNN (baseline رسمی Matbench) | 57.76 ± 12.31 | پایین‌ترین سطح مقایسه |
| Matformer | 42.53 ± 11.89 | |
| MODNet | 34.28 ± 2.08 | |
| M3GNet | 34.16 ± 4.5 | |
| ALIGNN | 29.54 ± 2.12 | |
| coGN | 29.71 ± 2.0 | |
| coNGN | 28.89 ± 3.28 | |
| DenseGNN (بهترین نسخه) | ~23.5 | از قوی‌ترین‌های منتشرشده |
| TriForces (fine-tune روی eSEN، پایه‌ی foundation model) | ~19.5 | نزدیک SOTA فعلی، خارج از دسته‌ی «from-scratch GNN» |

⚠️ این اعداد از جست‌وجوی وب (مقالات coGN/coNGN، DenseGNN، TriForces، و صفحه‌ی راهنمای
Matbench) جمع‌آوری شده‌اند، نه از خود سایت لیدربورد (که در دسترسی مستقیم مسدود بود).
عدد نهایی خودمان را باید در برابر همین بازه («baseline ضعیف» ~57، «GNN های قوی» ~29،
«SOTA فعلی» ~19-24) بسنجیم.


In [8]:
!git clone https://github.com/materialsproject/matbench.git

Cloning into 'matbench'...
remote: Enumerating objects: 10391, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 10391 (delta 97), reused 66 (delta 66), pack-reused 10270 (from 2)
Receiving objects: 100% (10391/10391), 368.98 MiB | 42.70 MiB/s, done.
Resolving deltas: 100% (6754/6754), done.


In [9]:
!ls matbench

benchmarks    docs_src	mkdocs.yml	requirements-dev.txt   setup.cfg
CITATION.cff  LICENSE	pyproject.toml	requirements-docs.txt  setup.py
docs	      matbench	README.md	scripts


In [10]:
!find matbench -maxdepth 2

matbench
matbench/pyproject.toml
matbench/docs
matbench/docs/assets
matbench/docs/sitemap.xml
matbench/docs/sitemap.xml.gz
matbench/docs/objects.inv
matbench/docs/static
matbench/docs/search
matbench/docs/stylesheets
matbench/docs/Reference
matbench/docs/Full Benchmark Data
matbench/docs/404.html
matbench/docs/Leaderboards Per-Task
matbench/docs/How To Use
matbench/docs/CNAME
matbench/docs/index.html
matbench/docs/Benchmark Info
matbench/scripts
matbench/scripts/test_submission.py
matbench/scripts/doc_snippets
matbench/scripts/mbv01_get_ds_stats.py
matbench/scripts/mvb01_generate_validation.py
matbench/scripts/release.py
matbench/scripts/mbv01
matbench/scripts/publish.sh
matbench/scripts/__init__.py
matbench/scripts/artifacts
matbench/scripts/rebuild_docs.py
matbench/scripts/coverage.py
matbench/requirements-docs.txt
matbench/LICENSE
matbench/docs_src
matbench/docs_src/static
matbench/docs_src/stylesheets
matbench/docs_src/Reference
matbench/docs_src/index.md
matbench/docs_src/Full Ben

In [11]:
import json

with open("matbench/matbench/matbench_v0.1_dataset_metadata.json") as f:
    meta = json.load(f)

print(meta.keys())

dict_keys(['matbench_dielectric', 'matbench_expt_gap', 'matbench_expt_is_metal', 'matbench_glass', 'matbench_jdft2d', 'matbench_log_gvrh', 'matbench_log_kvrh', 'matbench_mp_e_form', 'matbench_mp_gap', 'matbench_mp_is_metal', 'matbench_perovskites', 'matbench_phonons', 'matbench_steels'])


In [12]:
print(meta["matbench_phonons"])

{'input_type': 'structure', 'mad': 323.78696979348734, 'n_samples': 1265, 'target': 'last phdos peak', 'task_type': 'regression', 'unit': 'cm^-1'}


In [14]:
!grep -R "load_dataset" matbench/matbench

matbench/matbench/data_ops.py:from matminer.datasets import load_dataset
matbench/matbench/data_ops.py:    df = load_dataset(dataset_name)
matbench/matbench/metadata.py:from matminer.datasets.utils import _load_dataset_dict
matbench/matbench/metadata.py:MATMINER_DATASET_METADATA = _load_dataset_dict()


In [16]:
!pip install -U matminer

  Using cached scipy-1.18.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 65.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 60.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 86.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 117.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━

In [17]:
from matminer.datasets import load_dataset

df = load_dataset("matbench_phonons")
print(df.head())

Fetching matbench_phonons.json.gz from https://ml.materialsproject.org/projects/matbench_phonons.json.gz to /usr/local/lib/python3.12/dist-packages/matminer/datasets/matbench_phonons.json.gz


Fetching https://ml.materialsproject.org/projects/matbench_phonons.json.gz in MB: 0.4608MB [00:00, 214.22MB/s]                             


                                           structure  last phdos peak
0  [[2.8943817  2.04663693 5.01321616] Te, [0. 0....        98.585771
1  [[0.98372595 0.69559929 1.70386332] B, [0. 0. ...       701.585723
2  [[0. 0. 0.] Ba, [2.15053493 1.24161183 2.85808...      1138.585689
3  [[-2.23741407  0.         -2.23366548] Al, [0....       718.585722
4  [[1.60015264 0.92384464 2.65049608] B, [-8.649...       795.585716


In [18]:
!pip install torch_geometric

  Using cached torch_geometric-2.8.0-py3-none-any.whl.metadata (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.0 MB/s eta 0:00:0000:01


In [6]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [4]:
!pip install -q matbench torch_geometric
print("Installed: matbench, torch_geometric")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Installed: matbench, torch_geometric


In [20]:
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from tqdm.notebook import tqdm

from matminer.datasets import load_dataset
from sklearn.model_selection import KFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


## 1. Load the matbench_phonons task

In [21]:
df = load_dataset("matbench_phonons")

structures = df["structure"].tolist()
targets = df["last phdos peak"].to_numpy()

print(len(structures))

1265


## 2. Atomic feature table (same feature set as the main project: `5_geometry_radius_volume`)

⚠️ **نکته‌ی مهم:** این فیچرست در پروژه‌ی اصلی روی عناصر محدود فازهای MAX (M/A/X) تست شده
بود. دیتاست matbench_phonons عناصر بسیار متنوع‌تری دارد. زیر، پوشش این فیچرست روی عناصر
واقعی دیتاست matbench بررسی و گزارش می‌شود؛ اگر پوشش ضعیف بود، باید فیچرست جایگزین
(مثلاً از مندلیف/pymatgen مستقیم) در نظر گرفت.

In [22]:
FEATURE_CSV = '/kaggle/input/datasets/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"Atomic features: {N_ATOM_FEATURES} -> {feature_cols}")
print(f"Elements covered by feature table: {len(df_features.index)}")

Atomic features: 7 -> ['atomic_weight', 'atomic_radius', 'atomic_radius_rahm', 'covalent_radius_cordero', 'vdw_radius', 'atomic_volume', 'lattice_constant']
Elements covered by feature table: 118


In [23]:
# Check element coverage against the actual matbench_phonons dataset
train_inputs_fold0, _ = task.get_train_and_val_data(task.folds[0])
test_inputs_fold0 = task.get_test_data(task.folds[0], include_target=False)

all_structures_fold0 = list(train_inputs_fold0) + list(test_inputs_fold0)
all_elements = set()
for struct in all_structures_fold0:
    for site in struct:
        all_elements.add(site.specie.symbol)

missing_elements = sorted(all_elements - set(df_features.index))
covered_elements = sorted(all_elements & set(df_features.index))
print(f"Total unique elements in matbench_phonons (fold 0 sample): {len(all_elements)}")
print(f"Covered by feature table: {len(covered_elements)}")
print(f"Missing (will fall back to zero-vector): {len(missing_elements)}")
if missing_elements:
    print(f"Missing elements: {missing_elements}")

NameError: name 'task' is not defined

## 3. Graph construction (bond graph + atom graph) — same approach as the main project

In [ ]:
CUTOFF = 8.0


def structure_to_bond_graph(positions, n_atoms_sample):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def structure_to_atom_graph(atom_elements, positions, n_atoms_sample):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def pymatgen_structure_to_graphs(structure):
    positions = structure.cart_coords.astype(np.float32)
    atom_elements = [site.specie.symbol for site in structure]
    n_atoms = len(atom_elements)
    bond_g = structure_to_bond_graph(positions, n_atoms)
    atom_g = structure_to_atom_graph(atom_elements, positions, n_atoms)
    return bond_g, atom_g


print("Graph construction functions ready (reused pattern from the main project's notebook 15)")

## 4. Model: Dual Graph Regressor (بدون فیزیک، بدون Phonopy)

معماری انکودر دقیقاً همون الگوی دوگانه‌ی bond graph + atom graph پروژه‌ی اصلی است
(message passing + attention pooling + Set2Set/mean/max)، اما به‌جای سر پیش‌بینی
ماتریس IFC، یک MLP رگرسیون ساده خروجی یک عدد اسکالر (فرکانس پیک phonon DOS، بر حسب cm⁻¹)
می‌دهد.

In [ ]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphRegressor(nn.Module):
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1,
                 hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1,
                 set2set_steps=1):
        super().__init__()
        hidden = hidden_dim

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_bond_layers)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_bond_layers)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(n_atom_layers)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_atom_layers)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=set2set_steps)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        combined_dim = 8 * hidden + hidden // 4

        # Regression head: single scalar output (frequency at last phonon PhDOS peak, cm^-1)
        self.regression_head = nn.Sequential(
            nn.Linear(combined_dim, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.LayerNorm(64), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            global_features
        ], dim=1)

        return self.regression_head(combined).squeeze(-1)

print("DualGraphRegressor ready")

## 5. Training loop for a single fold

In [ ]:
def build_graphs(structures):
    bond_graphs, atom_graphs = [], []
    for s in tqdm(structures, desc="Building graphs", leave=False):
        bg, ag = pymatgen_structure_to_graphs(s)
        bond_graphs.append(bg)
        atom_graphs.append(ag)
    return bond_graphs, atom_graphs


def make_batches(n, batch_size, shuffle, rng):
    idx = np.arange(n)
    if shuffle:
        idx = rng.permutation(idx)
    for i in range(0, n, batch_size):
        yield idx[i:i+batch_size]


def run_epoch(model, bond_graphs, atom_graphs, targets, indices, optimizer, batch_size, shuffle, rng):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_batches = 0., 0
    for batch_idx in make_batches(len(indices), batch_size, shuffle, rng):
        actual_idx = [indices[i] for i in batch_idx]
        bond_list = [bond_graphs[i] for i in actual_idx]
        atom_list = [atom_graphs[i] for i in actual_idx]
        y = torch.tensor([targets[i] for i in actual_idx], dtype=torch.float32, device=device)

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            pred = model(bond_batch, atom_batch)
            loss = F.l1_loss(pred, y)  # MAE loss, matches the matbench evaluation metric
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def predict(model, bond_graphs, atom_graphs, batch_size=16):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(bond_graphs), batch_size):
            bond_list = bond_graphs[i:i+batch_size]
            atom_list = atom_graphs[i:i+batch_size]
            bond_batch = Batch.from_data_list(bond_list).to(device)
            atom_batch = Batch.from_data_list(atom_list).to(device)
            pred = model(bond_batch, atom_batch)
            preds.extend(pred.cpu().numpy().tolist())
    return preds


print("Training/prediction utilities ready")

## 6. اجرای کامل ۵-fold رسمی Matbench

برای هر fold:
1. گراف‌های bond/atom برای train+val (که matbench به‌صورت ترکیبی می‌دهد) و test ساخته می‌شود
2. یک split داخلی 90/10 برای early stopping اعمال می‌شود (matbench فقط train+val ترکیبی را
   می‌دهد؛ نحوه‌ی تقسیم داخلی به عهده‌ی خود ماست)
3. مدل با MAE loss (منطبق با معیار رسمی ارزیابی) آموزش داده می‌شود
4. پیش‌بینی روی test فولد با `task.record(fold, predictions)` ثبت می‌شود

⚠️ این حلقه از صفر برای هر fold آموزش می‌دهد (نه fine-tune از fold قبلی)، دقیقاً طبق
پروتکل nested cross-validation رسمی Matbench.

In [ ]:
EPOCHS = 300
PATIENCE = 40
BATCH_SIZE = 16
LR = 5e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.1
SEED = 42

fold_maes = []

for fold in task.folds:
    print(f"\n{'='*60}\nFold {fold}\n{'='*60}")

    train_val_inputs, train_val_outputs = task.get_train_and_val_data(fold)
    test_inputs = task.get_test_data(fold, include_target=False)

    train_val_structures = list(train_val_inputs)
    train_val_targets = list(train_val_outputs)
    test_structures = list(test_inputs)

    print(f"Train+val: {len(train_val_structures)} | Test: {len(test_structures)}")

    bond_graphs_tv, atom_graphs_tv = build_graphs(train_val_structures)
    bond_graphs_test, atom_graphs_test = build_graphs(test_structures)

    rng = np.random.default_rng(SEED + fold)
    n_tv = len(train_val_structures)
    perm = rng.permutation(n_tv)
    n_val = max(1, int(VAL_FRACTION * n_tv))
    val_idx = perm[:n_val].tolist()
    train_idx = perm[n_val:].tolist()

    torch.manual_seed(SEED + fold)
    model = DualGraphRegressor(
        n_bond_features=6, n_atom_features=N_ATOM_FEATURES, edge_dim=1,
        hidden_dim=128, n_bond_layers=5, n_atom_layers=2, dropout=0.1, set2set_steps=1
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        train_loss = run_epoch(model, bond_graphs_tv, atom_graphs_tv, train_val_targets,
                                train_idx, optimizer, BATCH_SIZE, True, rng)
        val_loss = run_epoch(model, bond_graphs_tv, atom_graphs_tv, train_val_targets,
                              val_idx, None, BATCH_SIZE, False, rng)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 20 == 0:
            print(f"  Epoch {epoch:4d} | Train MAE: {train_loss:.3f} | Val MAE: {val_loss:.3f} | Best: {best_val_loss:.3f}")

        if patience_counter >= PATIENCE:
            print(f"  Early stop at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/matbench_phonons_fold{fold}.pt')

    predictions = predict(model, bond_graphs_test, atom_graphs_test, batch_size=BATCH_SIZE)
    task.record(fold, predictions)

    fold_maes.append(best_val_loss)
    print(f"  Fold {fold} done. Best internal val MAE: {best_val_loss:.3f} cm^-1")

## 7. نتیجه‌ی نهایی رسمی (طبق پروتکل Matbench)

In [ ]:
mb.matbench_phonons.scores
print("Official matbench_phonons scores (5-fold nested CV):")
scores = mb.matbench_phonons.scores
print(f"  MAE mean: {scores['mae']['mean']:.3f} cm^-1")
print(f"  MAE std:  {scores['mae']['std']:.3f}")
print(f"  MAE min:  {scores['mae']['min']:.3f}")
print(f"  MAE max:  {scores['mae']['max']:.3f}")

print("\n=== Comparison with published leaderboard-area results ===")
print(f"  CGCNN (official Matbench baseline) : 57.76 cm^-1")
print(f"  Matformer                          : 42.53 cm^-1")
print(f"  MODNet                             : 34.28 cm^-1")
print(f"  M3GNet                             : 34.16 cm^-1")
print(f"  ALIGNN                             : 29.54 cm^-1")
print(f"  coGN                               : 29.71 cm^-1")
print(f"  coNGN                              : 28.89 cm^-1")
print(f"  DenseGNN (best variant)            : ~23.5 cm^-1")
print(f"  TriForces (eSEN foundation model)  : ~19.5 cm^-1")
print(f"  --------------------------------------------------")
print(f"  This project's Dual Graph GNN      : {scores['mae']['mean']:.2f} cm^-1")

In [ ]:
mb.to_file('/kaggle/working/matbench_phonons_results.json.gz')
print("Saved: /kaggle/working/matbench_phonons_results.json.gz")

## جمع‌بندی

این notebook مسیر مکملی نسبت به پروژه‌ی اصلی باز کرد: همون انکودر دوگانه‌ی گراف
(bond graph + atom graph) که برای پیش‌بینی IFC فازهای MAX ساخته شده بود، اینجا با یک
سر رگرسیون ساده روی تسک رسمی و عمومی‌تر `matbench_phonons` (پیش‌بینی یک عدد اسکالر —
فرکانس پیک آخر phonon DOS — برای 1265 ماده‌ی عمومی، نه فقط فازهای MAX) امتحان شد.

### نکات مهم برای تفسیر نتیجه
- این عدد MAE **قابل مقایسه‌ی مستقیم نیست** با MAE=0.909 (یا 0.8903 بعد از tuning) پروژه‌ی
  اصلی — آن عدد روی دیتاست دیگری (MAX Phase)، با واحد دیگر (THz)، و برای پیش‌بینی یک
  منحنی کامل dispersion (نه یک عدد) به‌دست آمده بود.
- پوشش فیچرست اتمی (`5_geometry_radius_volume`) روی عناصر واقعی matbench_phonons در
  بخش ۲ گزارش شد؛ اگر پوشش ناقص بود، این یک محدودیت شناخته‌شده است که می‌تواند نتیجه
  را بدتر از پتانسیل واقعی معماری نشان دهد.
- این اجرا از صفر و بدون هیچ tuning‌ای انجام شده (برخلاف notebook 21 که مفصل tuning شد)؛
  یعنی نتیجه‌ی این‌جا احتمالاً «حد پایین» عملکرد ممکن این معماری روی این تسک است، نه سقف آن.

### ثبت در Obsidian (بعد از اجرا روی Kaggle)
- عدد نهایی MAE (mean ± std روی ۵ فولد) و جایگاه آن نسبت به جدول بالا
- در صورت پوشش ضعیف فیچرست، بررسی جایگزینی با فیچرهای عمومی‌تر (مثل مندلیف کامل)
- تصمیم درباره‌ی ادامه یا نه: آیا این مسیر ارزش سرمایه‌گذاری بیشتر (tuning، فیچرهای بهتر) دارد
  یا فقط به‌عنوان یک نقطه‌ی مرجع/اعتبارسنجی جانبی برای مقاله کافی است؟
